# OULAD baseline model comparison

This notebook is an interactive view of the reusable OULAD pipeline. It does not contain the dataset or duplicate the cleaning and feature logic.

The current task is enrollment-level risk classification:

- `Fail` or `Withdrawn` → risk `1`
- `Pass` or `Distinction` → risk `0`

The features currently include student attributes, assessment aggregates, registration fields, and optional VLE activity aggregates. VLE clicks are activity counts, not study duration.

## Notebook map

This notebook assumes the data layer has been reviewed in `01_oulad_processing_and_features.ipynb`. It then creates the enrollment-level feature matrix, uses one fixed split for all models, compares the metrics, and records why the selected model is only a baseline.

## 1. Locate the repository and data

Place the OULAD CSV release in `data/raw/oulad/`. If processed tables already exist, this notebook uses them; otherwise it creates them with the project cleaner.

In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from sklearn.metrics import ConfusionMatrixDisplay

ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / 'src/data/cleaner.py').exists()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

RAW_DIR = ROOT / 'data/raw/oulad'
PROCESSED_DIR = ROOT / 'data/processed/oulad'
DATA_DIR = PROCESSED_DIR if (PROCESSED_DIR / 'studentInfo.csv').exists() else RAW_DIR

from src.data.cleaner import clean_oulad_tables, write_clean_tables
from src.data.loader import load_oulad_tables
from src.features.feature_engineering import build_oulad_features
from src.models.baseline import train_baselines
from src.models.baseline import split_baseline_data

print(f'Repository: {ROOT}')
print(f'Data directory: {DATA_DIR}')

In [ ]:
raw_tables = load_oulad_tables(DATA_DIR)
cleaned_tables = clean_oulad_tables(raw_tables)

if DATA_DIR == RAW_DIR:
    write_clean_tables(
        cleaned_tables,
        PROCESSED_DIR,
        raw_row_counts={name: len(frame) for name, frame in raw_tables.items()},
    )

table_summary = pd.DataFrame(
    [
        {'table': name, 'rows': len(frame), 'columns': len(frame.columns)}
        for name, frame in cleaned_tables.items()
    ]
).sort_values('table')
display(table_summary.reset_index(drop=True))

### Why these checks matter

The target is enrollment-level rather than student-level. The same student can appear in multiple course presentations. The class counts and missing-value checks are shown before training so that metric interpretation is tied to the actual population used.

## 2. Data-quality and outcome checks

Missing scores remain missing. They are not interpreted as zero. Invalid outcome labels are excluded from supervised training by the feature builder.

In [ ]:
student_info = cleaned_tables['studentInfo']
student_assessment = cleaned_tables['studentAssessment']

quality = pd.Series({
    'enrollments': len(student_info),
    'assessment_records': len(student_assessment),
    'known_scores': int(student_assessment['score'].notna().sum()),
    'missing_scores': int(student_assessment['score'].isna().sum()),
    'minimum_known_score': student_assessment['score'].min(),
    'maximum_known_score': student_assessment['score'].max(),
})
display(quality.to_frame('value'))
display(student_info['final_result'].value_counts(dropna=False).rename_axis('final_result').to_frame('enrollments'))

In [ ]:
# Colorful outcome distribution
outcome_counts = student_info['final_result'].value_counts().reindex(
    ['Distinction', 'Pass', 'Fail', 'Withdrawn']
)
colors = ['#2ca25f', '#66c2a4', '#fc8d62', '#e78ac3']
fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(outcome_counts.index, outcome_counts.values, color=colors, edgecolor='white', linewidth=1.5)
ax.set_title('OULAD enrollment outcomes', loc='left', fontsize=16, weight='bold')
ax.set_ylabel('Enrollments')
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='y', alpha=0.2)
for bar in bars:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), f'{int(bar.get_height()):,}',
            ha='center', va='bottom', fontsize=10)
fig.tight_layout()
plt.show()

### Model input boundary

The feature builder keeps identifiers and `final_result` for inspection, but the baseline model removes them. The model receives demographic/enrollment fields and aggregated assessment, registration, and optional activity features. Imputation and one-hot encoding happen inside each fitted pipeline after the train/test split.

## 3. Extract enrollment-level features

The feature builder joins assessment definitions to student assessment records, aggregates activity by enrollment, and creates the binary `target_risk`. Student and presentation identifiers are retained for inspection but removed from model inputs.

In [ ]:
features = build_oulad_features(cleaned_tables, clean=False)
print(f'Feature rows: {len(features):,}')
print(f'Feature columns: {len(features.columns):,}')
display(features.head())
display(features['target_risk'].value_counts().rename_axis('target_risk').to_frame('enrollments'))

In [ ]:
missingness = (
    features.isna().mean().sort_values(ascending=False).rename('missing_share').to_frame()
)
display(missingness[missingness['missing_share'] > 0].head(20))

In [ ]:
# Colorful missingness profile
missing_plot = missingness[missingness['missing_share'] > 0].head(12).sort_values('missing_share')
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(missing_plot.index, missing_plot['missing_share'] * 100,
        color=plt.cm.magma_r(range(40, 40 + 20 * len(missing_plot), 20)))
ax.set_title('Unknown values in extracted features', loc='left', fontsize=16, weight='bold')
ax.set_xlabel('Missing share (%)')
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='x', alpha=0.2)
fig.tight_layout()
plt.show()

### Why these four models

- Logistic regression is a transparent linear reference.
- Random forest provides a nonlinear bagging model.
- Extra trees adds stronger randomization and nonlinear interactions.
- Histogram gradient boosting is a compact boosting model for tabular data.

ROC-AUC evaluates ranking across thresholds, balanced accuracy gives equal weight to both classes, and F1 summarizes precision and recall at the classifier threshold. The selection rule is ROC-AUC first, then balanced accuracy, then F1.

## 4. Compare four baseline models

Every model uses the same deterministic 75/25 stratified holdout with `random_state=42`. Selection prioritizes ROC-AUC, then balanced accuracy, then F1.

In [ ]:
best_name, fitted_models, model_results = train_baselines(features)
comparison = (
    pd.DataFrame(model_results).T
    .sort_values(['roc_auc', 'balanced_accuracy', 'f1'], ascending=False)
)
display(comparison[['accuracy', 'balanced_accuracy', 'precision', 'recall', 'f1', 'roc_auc']])
print(f'Selected model: {best_name}')

In [ ]:
# Colorful model comparison
metric_names = ['roc_auc', 'balanced_accuracy', 'f1']
metric_labels = ['ROC-AUC', 'Balanced accuracy', 'F1']
metric_colors = ['#386cb0', '#fdb462', '#7fc97f']
fig, ax = plt.subplots(figsize=(11, 5.5))
x = range(len(comparison.index))
width = 0.24
for offset, metric, label, color in zip([-width, 0, width], metric_names, metric_labels, metric_colors):
    values = comparison[metric].astype(float).values
    ax.bar([position + offset for position in x], values, width, label=label, color=color)
ax.set_xticks(list(x), [name.replace('_', '\n') for name in comparison.index])
ax.set_ylim(0, 1.05)
ax.set_ylabel('Score')
ax.set_title('Baseline model comparison', loc='left', fontsize=16, weight='bold')
ax.legend(frameon=False, ncols=3)
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='y', alpha=0.2)
fig.tight_layout()
plt.show()

In [ ]:
# Assessment-score profile by risk label
score_by_risk = features.groupby('target_risk')['assessment_score_mean'].mean().reindex([0, 1])
score_labels = ['Pass/Distinction', 'Fail/Withdrawn']
fig, ax = plt.subplots(figsize=(7, 4.5))
bars = ax.bar(score_labels, score_by_risk.values, color=['#4daf4a', '#e41a1c'], width=0.6)
ax.set_title('Mean assessment score by risk group', loc='left', fontsize=15, weight='bold')
ax.set_ylabel('Mean assessment score')
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='y', alpha=0.2)
for bar in bars:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), f'{bar.get_height():.1f}',
            ha='center', va='bottom', fontsize=10)
fig.tight_layout()
plt.show()

### Confusion matrices

Each confusion matrix counts correct and incorrect predictions on the shared holdout. The rows are the actual labels and the columns are the predicted labels: 0 means Pass/Distinction and 1 means Fail/Withdrawn. These counts make the precision/recall trade-off visible, especially for risk screening.

In [ ]:
# Confusion matrices for all four fitted baselines
_, test_features, _, test_target = split_baseline_data(features, test_size=0.25, random_state=42)
fig, axes = plt.subplots(2, 2, figsize=(11, 9))
for ax, (model_name, model) in zip(axes.ravel(), fitted_models.items()):
    ConfusionMatrixDisplay.from_predictions(
        test_target,
        model.predict(test_features),
        display_labels=['Pass/Distinction', 'Fail/Withdrawn'],
        cmap='YlGnBu',
        colorbar=False,
        ax=ax,
    )
    ax.set_title(model_name.replace('_', ' ').title(), weight='bold')
fig.suptitle('Baseline confusion matrices', fontsize=17, weight='bold', x=0.5, y=0.98)
fig.tight_layout()
plt.show()

### Reading the model chart

The chart compares models on exactly the same holdout rows. A higher score indicates better performance for that metric on this split; it does not prove that the model will perform equally well on a future cohort. The selected model should be re-evaluated after a prediction cutoff and time-aware split are introduced.

## 5. Interpretation and limitations

The selected model is a benchmark for enrollment-risk prediction. The current features aggregate full-course assessment and VLE information while predicting the final outcome. That means these results are retrospective and may include information unavailable at an early prediction date.

The next modeling iteration should define a prediction cutoff, use time-aware features and splits, calibrate risk thresholds, and keep the predictive model separate from the study-schedule optimizer.